# 😊 PHILIA — Facial Emotion Fine-Tuning (Optuna)
**Model:** `google/vit-base-patch16-224` → fine-tuned on FER2013  
**Classes:** angry · disgust · fear · happy · neutral · sad · surprise  
**Optuna:** 8-trial hyperparameter search → final training with best params  
**Output:** saved to Google Drive → download and drop into PHILIA

**Runtime:** Set to `GPU` → Runtime > Change runtime type > T4 GPU

---
## Workflow (2 phases — both are needed!)
| Phase | Cells | What happens |
|-------|-------|--------------|
| **Phase 1 — Optuna search** | 1–9 | Runs 8 short trials to find the best learning rate, batch size, etc. |
| **Phase 2 — Final training** | 10–13 | Trains a *fresh* model end-to-end using those best hyperparameters. |

> ⚠️ **Both phases are required.** Phase 1 finds *what* to train with. Phase 2 does the real training.

## Key Improvements Over Baseline
- **Optuna hyperparameter search** — was hardcoded before; now adapts to your GPU/data
- **Focal Loss + Class Weights** — FER2013 is heavily imbalanced (happy >> disgust/fear)
- **Data Augmentation** — random flips, colour jitter, rotations help ViT generalise on faces
- **Macro F1 tracking** — reveals when minority emotion classes (disgust/fear) are failing
- **Label smoothing (0.1)** — prevents overconfidence on ambiguous expression labels
- **Gradient checkpointing** — allows larger effective batch size on T4
- **Early stopping (patience=3)** — stops before overfitting

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

In [ ]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────
!pip install -q transformers datasets accelerate evaluate scikit-learn Pillow optuna

In [ ]:
# ── Cell 3: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/PHILIA/models/facial_emotion'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Model will be saved to:', SAVE_DIR)

In [ ]:
# ── Cell 4: Load FER2013 from HuggingFace ──────────────────────────────────
from datasets import load_dataset
import collections

# FER2013: 7 emotions, pre-cropped 48x48 face images
# 28,709 train | 3,589 val | 3,589 test
print('Loading FER2013...')
fer = load_dataset('3una/Fer2013')
print('Splits:', fer)

# Get label names from dataset feature
label_feature = fer['train'].features.get('label')
label_names_raw = label_feature.names if hasattr(label_feature, 'names') else \
    ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']

print('Dataset label names:', label_names_raw)
print('Train dist:', dict(collections.Counter(fer['train']['label'])))

In [ ]:
# ── Cell 5: Map FER2013 labels to canonical + compute class weights ─────────
# FER2013 label mapping (handles variant names across dataset releases)
import collections, torch

CANONICAL = {
    'angry':    'angry',   'anger':    'angry',
    'disgust':  'disgust',
    'fear':     'fear',    'fearful':  'fear',
    'happy':    'happy',   'happiness':'happy',
    'sad':      'sad',     'sadness':  'sad',
    'surprise': 'surprise','surprised':'surprise',
    'neutral':  'neutral',
}
LABELS   = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

# Map integer label_id → raw name → canonical → final int
def remap_label(example):
    raw = label_names_raw[example['label']].lower()
    canonical = CANONICAL.get(raw, None)
    if canonical is None:
        return {'new_label': -1}   # will be filtered out
    return {'new_label': LABEL2ID[canonical]}

for split in fer:
    fer[split] = fer[split].map(remap_label)
    fer[split] = fer[split].filter(lambda x: x['new_label'] >= 0)

print('After remapping:')
for split in fer:
    cnt = dict(collections.Counter([LABELS[l] for l in fer[split]['new_label']]))
    print(f'  {split}: {len(fer[split])} samples | {cnt}')

# Compute class weights from training distribution to handle FER2013 imbalance
# FER2013 has ~13k happy samples vs ~400 disgust — huge imbalance!
label_counts  = collections.Counter(fer['train']['new_label'])
total         = sum(label_counts.values())
class_weights = torch.tensor(
    [total / (len(LABELS) * label_counts.get(i, 1)) for i in range(len(LABELS))],
    dtype=torch.float,
)
print('\nClass weights:', {LABELS[i]: f'{w:.3f}' for i, w in enumerate(class_weights)})

In [ ]:
# ── Cell 6: Feature extractor + Augmentation + Preprocessing ───────────────
# Augmentation helps ViT generalise to PHILIA's live webcam feed:
# - Random horizontal flip (faces are mostly symmetric)
# - Slight rotation (head tilts are common)
# - Colour jitter (webcam colour temp varies)
from transformers import ViTImageProcessor
from PIL import Image, ImageOps, ImageEnhance
import numpy as np
import random

MODEL_CHECKPOINT = 'google/vit-base-patch16-224'
processor = ViTImageProcessor.from_pretrained(MODEL_CHECKPOINT)

def augment_image(img: Image.Image, is_train: bool) -> Image.Image:
    """Apply augmentations only during training."""
    if not is_train:
        return img
    # Random horizontal flip
    if random.random() > 0.5:
        img = ImageOps.mirror(img)
    # Random rotation ±15°
    angle = random.uniform(-15, 15)
    img = img.rotate(angle, fillcolor=(128, 128, 128))
    # Random brightness jitter ×[0.8, 1.2]
    factor = random.uniform(0.8, 1.2)
    img = ImageEnhance.Brightness(img).enhance(factor)
    # Random contrast jitter ×[0.8, 1.2]
    factor = random.uniform(0.8, 1.2)
    img = ImageEnhance.Contrast(img).enhance(factor)
    return img

def make_preprocess_fn(is_train: bool):
    def preprocess(batch):
        images = []
        for img in batch['image']:
            if not isinstance(img, Image.Image):
                img = Image.fromarray(img)
            img = img.convert('RGB').resize((224, 224))
            img = augment_image(img, is_train)
            images.append(img)
        inputs = processor(images=images, return_tensors='np')
        inputs['labels'] = batch['new_label']
        return inputs
    return preprocess

print('Preprocessing training images (with augmentation)...')
train_ds = fer['train'].map(
    make_preprocess_fn(is_train=True), batched=True, batch_size=64,
    remove_columns=['image', 'label', 'new_label']
)

print('Preprocessing val/test images (no augmentation)...')
val_split = 'validation' if 'validation' in fer else 'test'
val_ds = fer[val_split].map(
    make_preprocess_fn(is_train=False), batched=True, batch_size=64,
    remove_columns=['image', 'label', 'new_label']
)
test_ds = fer['test'].map(
    make_preprocess_fn(is_train=False), batched=True, batch_size=64,
    remove_columns=['image', 'label', 'new_label']
)

train_ds.set_format('torch')
val_ds.set_format('torch')
test_ds.set_format('torch')
print(f'Done. Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

In [ ]:
# ── Cell 7: Metrics · FocalLossTrainer · model_init ────────────────────────
import evaluate, torch.nn as nn
import optuna
from sklearn.metrics import f1_score
from transformers import (
    ViTForImageClassification, Trainer, TrainingArguments, EarlyStoppingCallback
)

optuna.logging.set_verbosity(optuna.logging.WARNING)

accuracy_metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if hasattr(logits, '__len__') and not isinstance(logits, np.ndarray):
        import numpy as _np
        logits = _np.array(logits)
    preds  = np.argmax(logits, axis=-1)
    acc    = accuracy_metric.compute(predictions=preds, references=labels)['accuracy']
    f1     = f1_score(labels, preds, average='weighted')
    f1_mac = f1_score(labels, preds, average='macro')   # reveals minority-class failures
    return {'accuracy': acc, 'f1': f1, 'f1_macro': f1_mac}

FOCAL_GAMMA = 2.0   # γ=2 focuses on hard examples; bump to 3 if fear/disgust still bad

def focal_loss(logits, labels, gamma=FOCAL_GAMMA):
    """Focal loss with class weighting — handles FER2013's heavy happy/neutral bias."""
    weights = class_weights.to(logits.device)
    ce_loss = nn.functional.cross_entropy(logits, labels, weight=weights, reduction='none')
    pt = torch.exp(-ce_loss)
    return ((1 - pt) ** gamma * ce_loss).mean()

def model_init(trial=None):
    """Fresh ViT for every Optuna trial — required by hyperparameter_search."""
    return ViTForImageClassification.from_pretrained(
        MODEL_CHECKPOINT,
        num_labels=len(LABELS),
        label2id=LABEL2ID,
        id2label=ID2LABEL,
        ignore_mismatched_sizes=True,
    )

class FocalLossTrainer(Trainer):
    """Trainer with Focal Loss — critical for FER2013 class imbalance."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = focal_loss(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def hp_space(trial):
    """Optuna search space — image classification benefits from wide LR range."""
    return {
        'learning_rate':               trial.suggest_float('learning_rate', 5e-6, 5e-4, log=True),
        'per_device_train_batch_size': trial.suggest_categorical('per_device_train_batch_size', [32, 64]),
        'num_train_epochs':            trial.suggest_int('num_train_epochs', 5, 12),
        'weight_decay':                trial.suggest_float('weight_decay', 1e-4, 0.15, log=True),
        'warmup_ratio':                trial.suggest_float('warmup_ratio', 0.03, 0.20),
    }

print('model_init, FocalLossTrainer and hp_space defined.')

---
## Phase 1 — Optuna Hyperparameter Search
Runs **8 short trials** to discover the best learning rate, batch size, epochs, weight decay, and warmup ratio.
Each trial trains a fresh ViT for a few epochs; Optuna prunes poor trials early.

> **This does NOT produce your final model.** It only finds the best settings for Phase 2.

In [ ]:
# ── Cell 8: (Needed for compute_metrics) ──────────────────────────────────
# numpy must be imported in the same scope as compute_metrics uses it
import numpy as np

In [ ]:
# ── Cell 9: Optuna Hyperparameter Search (8 trials) ────────────────────────
# PHASE 1 — finds best hyperparameters, does NOT save a final model

search_args = TrainingArguments(
    output_dir='/content/optuna_facial_search',
    eval_strategy='epoch',
    save_strategy='no',          # no checkpoints during search
    logging_steps=100,
    fp16=True,
    gradient_checkpointing=True, # saves GPU memory
    report_to='none',
    remove_unused_columns=False, # required for ViT pixel_values column
)

search_trainer = FocalLossTrainer(
    model_init=model_init,       # model_init is REQUIRED here (not model=)
    args=search_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print('Running Optuna search — 8 trials × up to 12 epochs each...')
print('(Optuna will prune bad trials early — total time ~30-50 min on T4)')
best_run = search_trainer.hyperparameter_search(
    direction='maximize',
    backend='optuna',
    hp_space=hp_space,
    n_trials=8,
    compute_objective=lambda m: m['eval_f1'],   # optimise for weighted F1
)

print(f'\n✅ Phase 1 complete!')
print(f'   Best trial  : {best_run.run_id}')
print(f'   eval_f1     : {best_run.objective:.4f}')
print('   Best hyperparameters:')
for k, v in best_run.hyperparameters.items():
    print(f'     {k}: {v}')

---
## Phase 2 — Final Training with Best Hyperparameters
Now that we know the best settings, we train a **fresh ViT model all the way through** using those settings.
This is where the real production model is produced and saved.  
> **This is NOT a duplicate.** Phase 1 = tuning, Phase 2 = actual full training.

In [ ]:
# ── Cell 10: Final Training (PHASE 2) ─────────────────────────────────────
# Trains a BRAND NEW ViT using the best hyperparameters found by Optuna.
# This is the model that will be saved and used in PHILIA.

hp = best_run.hyperparameters

final_args = TrainingArguments(
    output_dir='/content/vit_facial_emotion',
    num_train_epochs=hp.get('num_train_epochs', 10),
    per_device_train_batch_size=hp.get('per_device_train_batch_size', 64),
    per_device_eval_batch_size=64,
    warmup_ratio=hp.get('warmup_ratio', 0.1),
    learning_rate=hp.get('learning_rate', 2e-5),
    weight_decay=hp.get('weight_decay', 0.01),
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=100,
    fp16=True,
    gradient_checkpointing=True,
    report_to='none',
    remove_unused_columns=False, # required for ViT pixel_values column
    label_smoothing_factor=0.1,  # prevents overconfidence on ambiguous expressions
)

final_trainer = FocalLossTrainer(
    model=model_init(),              # fresh model (no trial= arg needed here)
    args=final_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print('🚀 Starting Phase 2 — final training with best hyperparameters...')
final_trainer.train()

In [ ]:
# ── Cell 11: Evaluate on test set ─────────────────────────────────────────
from sklearn.metrics import classification_report

test_results = final_trainer.evaluate(test_ds)
print('Test results:', test_results)

pred_output = final_trainer.predict(test_ds)
preds = np.argmax(pred_output.predictions, axis=-1)
print(classification_report(
    pred_output.label_ids, preds,
    target_names=LABELS, digits=3
))

In [ ]:
# ── Cell 12: Save to Google Drive ─────────────────────────────────────────
print(f'Saving model to {SAVE_DIR} ...')
final_trainer.model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print('Done! Files in Drive:')
import os
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
    print(f'  {f}  ({size:.1f} MB)')

In [ ]:
# ── Cell 13: (Optional) Push to HuggingFace Hub ───────────────────────────
# from huggingface_hub import login
# login(token='YOUR_HF_TOKEN')
# final_trainer.model.push_to_hub('YOUR_HF_USERNAME/philia-facial-emotion')
# processor.push_to_hub('YOUR_HF_USERNAME/philia-facial-emotion')
print('Skipped (optional). Uncomment above lines to push to HF Hub.')